In [1]:
import torch
import numpy as np
import os
import cv2

In [2]:
# Erlaube Numpy-Rekonstruktion
#torch.serialization.add_safe_globals([np._core.multiarray._reconstruct, np.ndarray, np.dtype])

file_path = "./outputs_raw/tracking_results_3b558d5c-e59e-415b-add7-b9338b8d547f.pt"
output_label_dir = "./dataset/labels/train/video_01"
YOLO_CLASS_ID = 0


# Jetzt laden
data = torch.load(file_path, map_location='cpu', weights_only=False)

In [3]:
data[0].keys()
output_label_dir = "./dataset/labels/train/video_02"

In [7]:
for frame_idx in data:
    frame_data = data[frame_idx]
    
    obj_ids = frame_data['out_obj_ids']
    masks = frame_data['out_binary_masks'] # Shape: [Anzahl_Fische, 1, H, W]
    
    txt_filename = f"{(frame_idx + 1):04d}.txt"
    txt_path = os.path.join(output_label_dir, txt_filename)
    
    with open(txt_path, "w") as f:
        # Wir loopen über jeden einzelnen Fisch im aktuellen Frame
        for i in range(len(obj_ids)):
            # Maske für diesen speziellen Fisch extrahieren
            mask = masks[i].squeeze().astype(np.uint8)
            h, w = mask.shape
            
            # Konturen finden
            contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            for contour in contours:
                if len(contour) < 3:
                    continue 
                
                # Punkte normalisieren
                points = contour.reshape(-1, 2).astype(float)
                points[:, 0] /= w 
                points[:, 1] /= h 
                
                # In YOLO Zeile schreiben
                # Auch wenn es Fisch #5 ist, schreiben wir "0" für die Klasse "Fisch"
                flat_points = points.flatten()
                line = f"{YOLO_CLASS_ID} " + " ".join([f"{p:.6f}" for p in flat_points])
                
                f.write(line + "\n")

In [8]:
from ultralytics import YOLO
model = YOLO("yolo26n-seg.pt")  # load a pretrained model (recommended for training)


In [9]:
model.train(
    data='fish_data.yaml', 
    epochs=200,        # Wie oft der ganze Datensatz trainiert wird
    imgsz=640,         # Bildgröße (YOLO skaliert intern auf dieses Maß)
    batch=16,          # Wie viele Bilder gleichzeitig in den Speicher geladen werden
    device=0           # Nutze GPU 0 (falls vorhanden, sonst weglassen für CPU)
)

New https://pypi.org/project/ultralytics/8.4.83 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.12.0+cu126 CUDA:0 (NVIDIA GeForce RTX 2070 SUPER, 8192MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=fish_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=200, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0,

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x78733d3e1160>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041, 

In [7]:
import cv2
from ultralytics import YOLO

# 1. Modell laden
model = YOLO("runs/segment/train-6/weights/best.pt")

# 2. Videoquelle öffnen
video_path = "trackvideos/fishvideo2.mp4"
cap = cv2.VideoCapture(video_path)

# Video-Eigenschaften abrufen
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

# WICHTIG: Da wir um 90 Grad drehen, vertauschen wir width und height für den Output
out = cv2.VideoWriter('fish_counted_rotated.mp4', 
                         cv2.VideoWriter_fourcc(*'mp4v'), fps, (height, width))

print("Starte Tracking, Drehung und Zählung...")

# Da wir die Frames vor dem Tracking drehen wollen, nutzen wir eine manuelle Schleife
# anstatt model.track(source=...)
while cap.isOpened():
    success, frame = cap.read()
    if not success:
        break

    # --- SCHRITT 1: Frame um 90 Grad nach rechts drehen ---
    # cv2.ROTATE_90_CLOCKWISE dreht das Bild 90 Grad nach rechts
    frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

    # --- SCHRITT 2: Tracking auf dem gedrehten Frame ausführen ---
    # persist=True ist wichtig, damit der Tracker weiß, dass es sich um aufeinanderfolgende Frames handelt
    results = model.track(frame, persist=True, conf=0.1, iou=0.5, tracker="botsort.yaml")

    # Ergebnisse verarbeiten (ist eine Liste, wir nehmen den ersten Index)
    r = results[0]
    
    # Standard YOLO-Visualisierung (Boxen/Masken) auf den gedrehten Frame zeichnen
    annotated_frame = r.plot()
    
    # Anzahl der Fische zählen
    fish_count = len(r.boxes)
    
    # --- SCHRITT 3: Text auf den gedrehten Frame schreiben ---
    cv2.putText(
        annotated_frame, 
        f"Fische aktuell: {fish_count}", 
        (50, 50),                   
        cv2.FONT_HERSHEY_SIMPLEX,    
        1.5,                        
        (0, 255, 0),                
        3,                          
        cv2.LINE_AA
    )
    
    # In Datei schreiben
    out.write(annotated_frame)

# Ressourcen freigeben
cap.release()
out.release()
print("Fertig! Das gedrehte Video wurde als 'fish_counted_rotated.mp4' gespeichert.")

Starte Tracking, Drehung und Zählung...

0: 640x384 26 fishs, 34.8ms
Speed: 7.6ms preprocess, 34.8ms inference, 14.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 24 fishs, 11.4ms
Speed: 1.1ms preprocess, 11.4ms inference, 1.8ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 24 fishs, 8.7ms
Speed: 1.5ms preprocess, 8.7ms inference, 2.9ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 27 fishs, 7.2ms
Speed: 1.3ms preprocess, 7.2ms inference, 3.2ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 25 fishs, 8.7ms
Speed: 1.8ms preprocess, 8.7ms inference, 3.3ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 28 fishs, 7.4ms
Speed: 1.2ms preprocess, 7.4ms inference, 2.4ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 27 fishs, 7.5ms
Speed: 1.2ms preprocess, 7.5ms inference, 2.0ms postprocess per image at shape (1, 3, 640, 384)

0: 640x384 25 fishs, 7.2ms
Speed: 1.2ms preprocess, 7.2ms inference, 2.4ms postprocess

In [10]:
# 2. Führe Tracking auf einem Video aus
# source: Pfad zum Video
# save: Speichert das Ergebnis-Video
# show: Zeigt es live an (nur wenn ein Monitor angeschlossen ist)
# conf: Confidence-Threshold (nur Objekte über 0.3 Vertrauen tracken)
#model = YOLO("runs/segment/train-4/weights/best.pt") # Pfad zu DEINEM Training
results = model.track(
    source="trackvideos/fishvideo2.mp4", 
    save=True, 
    conf=0.1, 
    iou=0.5,
    tracker="botsort.yaml"  # oder "bytetrack.yaml"
)

print("Das Video wurde im Ordner 'runs/segment/track' gespeichert.")


WARNING ⚠️ 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/298) /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/trackvideos/fishvideo2.mp4: 384x640 31 fishs, 33.2ms
video 1/1 (frame 2/298) /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/trackvideos/fishvideo2.mp4: 384x640 25 fishs, 11.1ms
video 1/1 (frame 3/298) /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/trackvideos/fishvideo2.mp4: 384x640 27 fishs, 8.8ms
video 1/1 (frame 4/298) /mnt/c/Users/sinke/Desktop/Masterarbeit/yolo/trackvideos/fishvideo2.m